In [ ]:
# Date 10/18/2026
# Name:Anthony Flores
# Version 0.4
# SNHU CS-340-14096-M01 Client/Server Development

# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### Updated ### 
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

### Updated ###

username = "aacuser"
password = "YOUR PASSWORD GOES HERE"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)

#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#Add in Grazioso Salvare’s logo
### Updated ###
# path for logo - code_files/Grazioso Salvare Logo.png
image_filename = 'Grazioso Salvare Logo.png' # added Grazioso Salvare logo image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

###Updated###
app.layout = html.Div([
    html.Center([
        html.H1('Grazioso Salvare Rescue Dashboard'),
        html.H3('Anthony Flores - October 18th, 2026'), #unique identifier such as your name or date
        html.A(
            html.Img( #HTML image tag
                src='data:image/png;base64,{}'.format(
                    encoded_image.decode()
                ),
                style={'height': '100px'}
            ),
            href='https://www.snhu.edu',
            target='_blank'
        )
    ]),
    html.Hr(),
    html.Div([
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'Water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'Mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'Disaster'},
                {'label': 'Reset', 'value': 'Reset'}
            ],
            value='Reset',
            labelStyle={'display': 'inline-block', 'marginRight': '20px'}
        )
    ]),
#Added code for the interactive filtering options. For example, Radio buttons, drop down, checkboxes, etc.
### Updated ###
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
                         editable=False,
                         filter_action="native",
                         sort_action="native",
                         sort_mode="multi",
                         page_action="native",
                         page_current=0,
                         page_size=10,
                         row_selectable="single",
                         selected_rows=[0],
### Updated ###
                        ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################

    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])

def update_dashboard(filter_type):
    # Water Rescue
    if filter_type == 'Water':
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland"
                ]
            },
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    # Mountain or Wilderness Rescue
    elif filter_type == 'Mountain':
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }
    # Disaster or Invidual Tracking
    elif filter_type == 'Disaster':
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Doberman Pinscher", 
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 20,
                "$lte": 300
            }
        }
    # Reset (return all records)
    else:
        query ={}
    # Retrieve matching records from MongoDB
    results = db.read(query)
    # Convert results into a DataFrame
    dff = pd.DataFrame.from_records(results)
    # Remove MongoDB ObjectId column
    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)
    # Return records to dashboard table
    return dff.to_dict('records')
###Updated###
# Display the breeds of animal based on quantity represented in
# the data table

@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    if viewData is None:
        dff = df.copy()
    else:
        dff = pd.DataFrame.from_records(viewData)
    if dff.empty:
        return [html.P("No animals found from filter(s) selected.")]    
    # Count number of animals by breed
    breed_counts = dff['breed'].value_counts().head(10)

    ### Updated added pie chart ###
    fig = px.pie(
        names=breed_counts.index,
        values=breed_counts.values,
        title='Common Breeds in area'
    )
    
    return [
        dcc.Graph(figure=fig)
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if not selected_columns:
        return []
    
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None:
        dff = df.copy()
    else:
        dff = pd.DataFrame.from_records(viewData)
    if dff.empty:
        return [html.P("No animal locations available.")]
    
    # Default to first animal if none selected
    if not index or index[0] >= len(dff):
        row = 0
    else: 
        row = index[0]
        
    # Center geolocation map on Austin TX [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Create map marker using selected animal's coordinates
            # Display animal's breed in the tooltip and name in the group
            dl.Marker(position=[
                dff.iloc[row]['location_lat'],
                dff.iloc[row]['location_long']
            ],
            children=[
                dl.Tooltip(str(dff.iloc[row]['breed'])),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(str(dff.iloc[row]['name']))
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 